In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
#======= Read in Performance Metrics Files =======
summary = pd.read_csv('../../../CO2-H2S/Data_Size_Test/Summary_metrics.csv')
# actual initial size (n_iters counts the first fit): PX 35 -> 36, PXT 15 -> 20 and 30 -> 40
summary['n_init'] = (summary.n_train_final_mean - summary.n_iters_mean + 1).round().astype(int)
# PXT 15 = 20 and 30 = 40 are the same run: keep one
summary = summary.drop_duplicates(['run', 'space', 'n_init']).sort_values('n_init')
# standard error of the mean over the 5 data splits
summary['MAPE_se'] = summary.MAPE_std / np.sqrt(summary.n_seeds)
summary['n_iters_mean'] = summary.n_iters_mean.round().astype(int)

def metrics(run, space):
    # same column order as the old Metrics files (size, R2, MAE, MAPE, iterations) + MAPE standard error
    d = summary[(summary.run == run) & (summary.space == space)]
    return d[['n_init', 'R2_mean', 'MAE_mean', 'MAPE_mean', 'n_iters_mean', 'MAPE_se']].reset_index(drop=True)

In [ ]:
#======= Define Plot Values and Generate Plots ========
# P-X Dataframes
HS_px = metrics('TAL_Pure_HS_HS_M', 'px')
print(HS_px)
co_px = metrics('TAL_Pure_CO_CO_M', 'px')

# P-X-T Dataframes
HS_pxt = metrics('TAL_Pure_HS_HS_M', 'pxt')
co_pxt = metrics('TAL_Pure_CO_CO_M', 'pxt')

# P-X Values
data_size_px_hs = HS_px.iloc[:, 0]
data_size_px_co = co_px.iloc[:, 0]

# CO2 metrics
mape_co2_px = co_px.iloc[:, 3]
no_iters_co2_px = co_px.iloc[:, 4]

# H2S metrics
mape_h2s_px = HS_px.iloc[:, 3]
no_iters_h2s_px = HS_px.iloc[:, 4]

# P-X-T Values
data_size_pxt_hs = HS_pxt.iloc[:, 0]
data_size_pxt_co = co_pxt.iloc[:, 0]

# CO2 metrics
mape_co2_pxt = co_pxt.iloc[:, 3]
no_iters_co2_pxt = co_pxt.iloc[:, 4]

# H2S metrics
mape_h2s_pxt = HS_pxt.iloc[:, 3]
no_iters_h2s_pxt = HS_pxt.iloc[:, 4]

# Standard Error Values
se_co2_px = co_px.iloc[:, 5]
se_h2s_px = HS_px.iloc[:, 5]

se_co2_pxt = co_pxt.iloc[:, 5]
se_h2s_pxt = HS_pxt.iloc[:, 5]


# Generate Plot
panel_labels = ["(a)", "(b)"]
fig, ax = plt.subplots(figsize=(15, 6), ncols=2)

scale = 3  # adjust bubble size

# P-X Plot
for i, ds in enumerate(data_size_px_co):
    ax[0].scatter(ds, mape_co2_px[i]*100, s=no_iters_co2_px[i] * scale, 
               color='steelblue', alpha=0.5, zorder=5, edgecolors = 'darkblue')
for i, ds in enumerate(data_size_px_hs):
    ax[0].scatter(ds, mape_h2s_px[i]*100, s=no_iters_h2s_px[i] * scale, 
               color='darkgreen', alpha=0.5, zorder=5, edgecolors = 'black')

# Connect bubbles with a thin line to show trend
ax[0].plot(data_size_px_co, mape_co2_px*100, color='steelblue',  linestyle='--', linewidth=2.0, label=r'$\mathrm{CO}_{2}$')
ax[0].plot(data_size_px_hs, mape_h2s_px*100, color='darkgreen', linestyle='--', linewidth=2.0, label=r'$\mathrm{H}_{2}{S}$')

ax[0].errorbar(data_size_px_co, mape_co2_px*100, yerr=se_co2_px*100, fmt='none', ecolor='darkblue',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[0].errorbar(data_size_px_hs, mape_h2s_px*100, yerr=se_h2s_px*100, fmt='none', ecolor='black',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[0].margins(x=0.06, y=0.10)

# Size legend
for iters in [min(no_iters_co2_px), max(no_iters_h2s_px)]:
    ax[0].scatter([], [], s=iters * scale, color='grey', alpha=0.5,
               label=f'{iters} TAL iters')

ax[0].set_xlabel('Initial Training Data Size', fontsize = 15, fontweight="bold")
ax[0].set_ylabel('MAPE (%)', fontsize = 15, fontweight="bold")
for tick in ax[0].get_xticklabels() + ax[0].get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")
# ax[0].set_title('TAL Performance vs Starting Mixture Data Size - P-X Space', fontsize = 15)
ax[0].legend(loc='upper right', fontsize = 12)
ax[0].grid(True, linestyle='--', alpha=0.4)
ax[0].text(-0.10,1.02,panel_labels[0],transform=ax[0].transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
        clip_on=False)

# P-X-T Plot
for i, ds in enumerate(data_size_pxt_co):
    ax[1].scatter(ds, mape_co2_pxt[i]*100, s=no_iters_co2_pxt[i] * scale, 
               color='steelblue', alpha=0.5, zorder=5, edgecolors = 'darkblue')
for i, ds in enumerate(data_size_pxt_hs):
    ax[1].scatter(ds, mape_h2s_pxt[i]*100, s=no_iters_h2s_pxt[i] * scale, 
               color='darkgreen', alpha=0.5, zorder=5, edgecolors = 'black')

# Connect bubbles with a thin line to show trend
ax[1].plot(data_size_pxt_co, mape_co2_pxt*100, color='steelblue',  linestyle='--', linewidth=2.0, label=r'$\mathrm{CO}_{2}$')
ax[1].plot(data_size_pxt_hs, mape_h2s_pxt*100, color='darkgreen', linestyle='--', linewidth=2.0, label=r'$\mathrm{H}_{2}{S}$')

ax[1].errorbar(data_size_pxt_co, mape_co2_pxt*100, yerr=se_co2_pxt*100, fmt='none', ecolor='darkblue',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[1].errorbar(data_size_pxt_hs, mape_h2s_pxt*100, yerr=se_h2s_pxt*100, fmt='none', ecolor='black',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[1].margins(x=0.06, y=0.10)

# Size legend
for iters in [min(no_iters_co2_pxt), max(no_iters_h2s_pxt)]:
    ax[1].scatter([], [], s=iters * scale, color='grey', alpha=0.5,
               label=f'{iters} TAL iters')

ax[1].set_xlabel('Initial Training Data Size', fontsize = 15, fontweight="bold")
ax[1].set_ylabel('MAPE (%)', fontsize = 15, fontweight="bold")
for tick in ax[1].get_xticklabels() + ax[1].get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")
# ax[1].set_title('TAL Performance vs Starting Mixture Data Size - P-X-T Space', fontsize = 15)
ax[1].legend(loc='best', fontsize = 12)
ax[1].grid(True, linestyle='--', alpha=0.4)
ax[1].text(-0.10,1.02,panel_labels[1],transform=ax[1].transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
        clip_on=False)
plt.tight_layout()
plt.savefig('Data_size_test.png', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.savefig('Data_size_test.pdf', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.show()